# Friends Demo — Phase 2: RAG Generation & Layer Scoping

Connecting memory retrieval to LLM answer generation and multi-entity scope management.


### 📋 Phase 2 Overview & Execution Flow

Phase 2 demonstrates how memories power LLM responses and multi-entity scoping:

- **Section 0 — Reconnect & Audit**: Reconnects to `MemoryClient` and inspects starting memories for Maya, Jordan, and Sam.
- **Section 1 — RAG Generation (`answer_with_memory`)**: Retrieves memories for a query, injects them into the LLM system prompt, and generates answers.
- **Section 2 — Conflict Resolution**: Observes how Mem0 handles contradictions when Maya switches hobbies from pottery to painting.
- **Section 3 — Multi-Entity Layer Scoping**: Demonstrates memory segregation across `user_id` (personal facts), `agent_id` (system rules), and `run_id` (session context).


In [1]:
import os
from dotenv import load_dotenv
from mem0 import MemoryClient

load_dotenv()
client = MemoryClient(api_key=os.getenv("MEM0_API_KEY"))

for user_id in ["maya", "jordan", "sam"]:
    print(f"--- {user_id} ---")
    memories = client.get_all(filters={"user_id": user_id})
    for item in memories.get("results", []):
        print(" -", item["memory"])
    print()

--- maya ---
 - Maya went hiking on the weekend of July 31 to August 1, 2026
 - Maya's pottery class moved from Tuesdays to Thursdays starting next month.
 - User recently started attending a pottery class that takes place on Tuesdays
 - User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing

--- jordan ---
 - Jordan wants to try a new ramen recipe on the weekend of August 6–7, 2026
 - User is planning a jazz-themed dinner party for September 2026 and wants recipe ideas
 - I spent the whole afternoon cooking a big Italian dinner for friends. I also play jazz piano most evenings to unwind.
 - User plays jazz piano most evenings to unwind
 - User spent the afternoon of August 5, 2026 cooking a big Italian dinner for friends

--- sam ---
 - Got really into this new video game this week. Also, heads up, I'm allergic to peanuts, so no peanut sauce next time.
 - User is allergic to peanuts and requests no peanut sauce in future meals
 - User got really into a

## 1. Generation -- answer a question using memory

mem0 retrieves facts, it doesn't answer questions. We still need our own LLM call for that,
using the lab's config from earlier notebooks.


In [2]:
from lab_llm_config import complete

def answer_with_memory(question, user_id):
    results = client.search(query=question, filters={"user_id": user_id})
    memories = [r["memory"] for r in results.get("results", [])]

    memory_block = "\n".join(f"- {m}" for m in memories)
    prompt = (
        "You're chatting with a friend. Use these known facts about them if relevant, "
        "and answer naturally without mentioning 'stored memories':\n\n"
        f"{memory_block}\n\nQuestion: {question}"
    )
    answer = complete(prompt)
    return answer, memories

<frozen abc>:106: DeprecationWarning: BaseAgentConfig is deprecated and will be removed in future versions. Config is now loaded via reflection so the separate config class is no longer needed.


In [3]:
question = "What should I get Maya for her birthday?"
answer, used_memories = answer_with_memory(question, user_id="maya")

print("Memories used:")
for m in used_memories:
    print(" -", m)

print("\nAnswer:")
print(answer)

Memories used:
 - Maya went hiking on the weekend of July 31 to August 1, 2026
 - Maya's pottery class moved from Tuesdays to Thursdays starting next month.
 - User recently started attending a pottery class that takes place on Tuesdays
 - User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing

Answer:
Since Maya’s been out on the trails and is now juggling a Thursday pottery class, a gift that hits both passions would feel extra thoughtful. Here are a few ideas that blend hiking and pottery:

| Idea | Why it fits | How to personalize |
|------|-------------|--------------------|
| **Premium hiking daypack** | She just did the Rockies, so a lightweight, ergonomic pack will make future hikes even more enjoyable. | Add a custom patch or her initials. |
| **Hydration‑bladder or trekking‑water bottle** | Keeps her hydrated on long treks


### Store this exchange, same as any other turn

In [4]:
client.add(
    [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ],
    user_id="maya",
)
print("Stored.")

Stored.


## 2. Conflict resolution -- Maya changes hobbies

She stops doing pottery and takes up painting instead. Snapshot before and after, so the
change (or lack of one) is visible, not assumed.


In [5]:
before = client.get_all(filters={"user_id": "maya"})
before_facts = [item["memory"] for item in before.get("results", [])]

In [6]:
add_result = client.add(
    "Actually, I quit pottery a few weeks ago -- I've switched to painting instead, "
    "I go to a studio on Thursdays now.",
    user_id="maya",
)
print(add_result)

{'event_id': 'c9ad01a6-267f-466c-8503-dc0f933f327e', 'status': 'PENDING'}


In [7]:
after = client.get_all(filters={"user_id": "maya"})
after_facts = [item["memory"] for item in after.get("results", [])]

print("BEFORE:")
for f in before_facts:
    print(" -", f)

print("\nAFTER:")
for f in after_facts:
    print(" -", f)

BEFORE:
 - Maya went hiking on the weekend of July 31 to August 1, 2026
 - Maya's pottery class moved from Tuesdays to Thursdays starting next month.
 - User recently started attending a pottery class that takes place on Tuesdays
 - User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing

AFTER:
 - Maya went hiking on the weekend of July 31 to August 1, 2026
 - Maya's pottery class moved from Tuesdays to Thursdays starting next month.
 - User recently started attending a pottery class that takes place on Tuesdays
 - User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing


### Confirm at the retrieval level

Does search correctly favor painting over pottery when asked directly?


In [8]:
results = client.search(query="what hobby is Maya doing these days?", filters={"user_id": "maya"})
for r in results.get("results", []):
    print(f"{r['score']:.3f}  {r['memory']}")

0.401  User is looking for birthday gift ideas for Maya
0.399  Assistant recommended a premium hiking daypack, which can be personalized with a custom patch or Maya's initials, as a birthday gift for Maya
0.389  Maya went hiking on the weekend of July 31 to August 1, 2026
0.323  Maya's pottery class moved from Tuesdays to Thursdays starting next month.
0.302  Assistant recommended a hydration‑bladder or trekking‑water bottle to keep Maya hydrated on long treks as a birthday gift
0.240  User recently started attending a pottery class that takes place on Tuesdays
0.203  User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing


Look at what's actually there before drawing a conclusion. If the pottery fact is still
sitting in storage unchanged, that's the same finding as the aircraft example -- storage adds,
it doesn't reliably reconcile. Whatever these three cells show is the real answer.


## 3. Layer scoping with `user_id` / `run_id` / `agent_id`

| Layer | mem0 filter |
|---|---|
| User (durable) | `user_id` only |
| Session (ephemeral) | `user_id` **and** `run_id` |
| Agent (behavior rule) | `agent_id` |

### 3a. A session-scoped fact

Something that only matters to today's conversation with Jordan, not their permanent record.


In [9]:
client.add(
    "Jordan is stressed today about planning the jazz-themed dinner party.",
    user_id="jordan",
    run_id="chat_today",
)
print("Session-scoped fact stored under run_id='chat_today'.")

Session-scoped fact stored under run_id='chat_today'.


### 3b. Does a plain `user_id` search pick it up?

In [10]:
results_user_only = client.search(
    query="how is Jordan feeling about the dinner party",
    filters={"user_id": "jordan"},
)
print("user_id only:")
for r in results_user_only.get("results", []):
    print(" -", r["memory"])

user_id only:
 - User is planning a jazz-themed dinner party for September 2026 and wants recipe ideas
 - Jordan wants to try a new ramen recipe on the weekend of August 6–7, 2026
 - I spent the whole afternoon cooking a big Italian dinner for friends. I also play jazz piano most evenings to unwind.
 - User spent the afternoon of August 5, 2026 cooking a big Italian dinner for friends
 - User plays jazz piano most evenings to unwind


In [11]:
results_with_run = client.search(
    query="how is Jordan feeling about the dinner party",
    filters={"AND": [{"user_id": "jordan"}, {"run_id": "chat_today"}]},
)
print("user_id + run_id:")
for r in results_with_run.get("results", []):
    print(" -", r["memory"])

user_id + run_id:
 - Jordan is stressed on August 5, 2026 while planning the jazz-themed dinner party


### 3c. An agent-layer fact

Not about any specific friend -- a behavior rule for the assistant itself. Added standalone
since it doesn't naturally arise from the conversation.

**Heads up:** combining `user_id` + `agent_id` filters together is a known flaky case in mem0
-- run the cells below and see what you actually get.


In [12]:
client.add(
    "Always suggest specific, personal gift ideas -- never generic suggestions like 'a gift card'.",
    agent_id="friend_assistant",
)
print("Agent-layer fact stored.")

Agent-layer fact stored.


In [14]:
by_agent_only = client.get_all(filters={"agent_id": "friend_assistant"})
print("agent_id only:")
for item in by_agent_only.get("results", []):
    print(" -", item["memory"])

agent_id only:
 - User prefers specific, personal gift ideas and requests that generic suggestions like gift cards be avoided


In [18]:
by_user_and_agent = client.get_all(
    filters={"AND": [{"user_id": "maya"}, {"agent_id": "friend_assistant"}]}
)
print("user_id + agent_id combined:")
for item in by_user_and_agent.get("results", []):
    print(" -", item["memory"])
print("\n(Empty here matches the known mem0 filter-combination issue -- not a bug in your code.)")

user_id + agent_id combined:

(Empty here matches the known mem0 filter-combination issue -- not a bug in your code.)


In [22]:
client.add(
    "Maya asked for a birthday gift suggestion.",
    user_id="maya",
    agent_id="friend_assistant",
)

# Now try the combined filter again
by_user_and_agent = client.get_all(
    filters={"AND": [{"user_id": "maya"}, {"agent_id": "friend_assistant"}]}
)
print(by_user_and_agent.get("results", []))

[]


In [23]:
# 1. Does the memory even exist with user_id alone?
by_user = client.get_all(filters={"user_id": "maya"})
print("user_id only:", [item["memory"] for item in by_user.get("results", [])])

# 2. Does it exist with agent_id alone?
by_agent = client.get_all(filters={"agent_id": "friend_assistant"})
print("agent_id only:", [item["memory"] for item in by_agent.get("results", [])])

# 3. The combined AND filter that returned []
by_both = client.get_all(
    filters={"AND": [{"user_id": "maya"}, {"agent_id": "friend_assistant"}]}
)
print("combined:", [item["memory"] for item in by_both.get("results", [])])

user_id only: ['Maya asked the user for a birthday gift suggestion', 'User quit pottery a few weeks ago (around July 15, 2026) and switched to painting, attending a studio on Thursdays', 'Assistant recommended a hydration‑bladder or trekking‑water bottle to keep Maya hydrated on long treks as a birthday gift', 'User is looking for birthday gift ideas for Maya', "Assistant recommended a premium hiking daypack, which can be personalized with a custom patch or Maya's initials, as a birthday gift for Maya", 'Maya went hiking on the weekend of July 31 to August 1, 2026', "Maya's pottery class moved from Tuesdays to Thursdays starting next month.", 'User recently started attending a pottery class that takes place on Tuesdays', 'User went hiking in the Rockies on the weekend of August 1-2, 2026 and found it amazing']
agent_id only: ['User prefers specific, personal gift ideas and requests that generic suggestions like gift cards be avoided']
combined: []


In [25]:
client.add(
    "Maya asked the assistant for a birthday gift suggestion.",
    user_id="maya",
    metadata={"agent_context": "friend_assistant"},
)

# Then filter like this instead
client.get_all(filters={"AND": [{"user_id": "maya"}, {"metadata": {"agent_context": "friend_assistant"}}]})

{'count': 1,
 'next': None,
 'previous': None,
 'results': [{'id': '2f1f3565-3b6a-4da8-8750-67bcdd1c2659',
   'memory': 'Maya asked the assistant for a birthday gift suggestion',
   'user_id': 'maya',
   'metadata': {'agent_context': 'friend_assistant'},
   'categories': None,
   'created_at': '2026-08-04T18:10:14-07:00',
   'updated_at': '2026-08-04T18:10:18-07:00',
   'expiration_date': None,
   'structured_attributes': {'year': 2026,
    'month': 8,
    'day': 5,
    'hour': 1,
    'minute': 10,
    'day_of_week': 'wednesday',
    'week_of_year': 32,
    'day_of_year': 217,
    'quarter': 3,
    'is_weekend': False},
   'replaced_by': None,
   'synthesized': False}]}

## Wrap-up

1. **Generation** turns retrieval into an actual usable answer.
2. Maya's pottery-to-painting switch is the same test as the aircraft engineer's fleet change
   -- whatever it showed here should match what you found there, since it's testing the same
   mechanism against a different story.
3. `run_id` and `agent_id` scoping behave the way this notebook's cells actually showed --
   not the way the docs describe in the abstract.
